# GAT net ordering + negotiated router: train on Colab

The GAT decides the order nets are routed in; the toolkit's negotiated router (Dijkstra with congestion costs) connects them. Training uses the challenge score directly: loss = total delay / baseline delay (= 1 / score), with its gradient taken through the "sort + route" solver by routing noisy versions of the GAT's order (perturbed optimizers).

This notebook sets everything up, checks it, trains, scores the result on the hard tier, and then **shuts the runtime down so it stops using compute units**.

**How to use it**
1. **Runtime → Change runtime type**: a GPU isn't needed; **more CPU cores** make it faster (every step routes `boards × samples` boards with the negotiated router, ~10–45 s each on hard-tier sizes).
2. Fill in the **Settings** cell below (at least `GAT_REPO_URL`).
3. **Runtime → Run all**. Approve the Google Drive pop-up at the start; after that it runs unattended.

**What happens, in order**

| step | time (roughly) |
|---|---|
| setup: Drive, clone the challenge toolkit + your repo, install PyTorch Geometric | 1–2 min |
| self-test: every step at tiny scale | 5–8 min |
| training data: the hard boards of your existing `gat_mcts_work/data_tiers` pool, or a new pool (only the first time) | 0 min / 1–3 h |
| training: until the eval score plateaus (`PLATEAU_EVALS`) | no time limit |
| evaluation: all 9 hard-tier cases, leaderboard-format submission | 20–60 min |
| report + automatic shutdown | – |

**Budget safety**
* If any step fails, the error is printed and the runtime is shut down (when `AUTO_SHUTDOWN = True`).
* There is no hour limit: stop by hand (**Runtime → Interrupt execution**) to end sooner; the latest checkpoint is kept.
* If Colab disconnects, **Run all** again with the same `RUN_NAME`: training resumes from the last checkpoint.
* Everything is saved in Google Drive under `MyDrive/gat_order_work/`.

## Settings

In [ ]:
# ---- edit these ----------------------------------------------------------------
GAT_REPO_URL = "https://github.com/YOUR_USER/YOUR_REPO.git"  # your repo containing the gat_order/ folder
GAT_REPO_BRANCH = "main"
GAT_REPO_SUBDIR = ""            # folder inside the repo that contains gat_order/ ("" = repo root)
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"  # private repo only: name of the Colab secret
                                      # (key icon in the left bar) holding a GitHub token

RUN_NAME = "order1"             # checkpoints/logs go to MyDrive/gat_order_work/<RUN_NAME>
PLATEAU_EVALS = 5               # stop once the eval score hasn't improved in this many
                                # evaluations (one every 10 steps); the only stop, so >= 1
EXISTING_POOL = "/content/drive/MyDrive/gat_mcts_work/data_tiers"  # its hard boards are used if it exists
N_TRAIN_BOARDS = 300            # size of a new pool, if there is no existing one
RUN_SELFTEST = True             # a few minutes' check before training; leave on
EVALUATE = True                 # score all hard-tier cases after training
EVAL_SAMPLES = 16               # orders routed per case when scoring: the GAT's + noisy ones
AUTO_SHUTDOWN = True            # disconnect the runtime at the end or on any error
# ---------------------------------------------------------------------------------

CHALLENGE_URL = "https://github.com/partcleda/eda-3d-routing-challenge.git"
CHALLENGE_COMMIT = "49b6443d9b599b77e09432c379ea7f422bdf4ffc"  # the version gat_order was tested on
WORK = "/content/drive/MyDrive/gat_order_work"
CHALLENGE_DIR = "/content/eda-3d-routing-challenge"
REPO_DIR = "/content/gat_repo"
DATA_LOCAL = "/content/order_data"

import os, sys, json, shutil, subprocess, traceback

if PLATEAU_EVALS < 1:
    raise ValueError("PLATEAU_EVALS must be >= 1: there is no hour limit, so it is the only thing that stops training.")
if "YOUR_USER" in GAT_REPO_URL:
    raise ValueError("Set GAT_REPO_URL in the Settings cell to the repo that holds gat_order/, then Run all again.")

ENV = dict(os.environ, GIT_TERMINAL_PROMPT="0", M3D_CHALLENGE_ROOT=CHALLENGE_DIR)

def sh(cmd, cwd=None, show=None):
    # Run a command, streaming its output into this cell; raise if it fails.
    print("$", show or cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=ENV, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout:
            if "UserWarning" in line or line.lstrip().startswith("warnings.warn"):
                continue
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        p.terminate()
        raise
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {show or cmd}")

def stop_runtime(reason):
    print(f"\n=== {reason} ===", flush=True)
    if not AUTO_SHUTDOWN:
        print("AUTO_SHUTDOWN is False: the runtime keeps running (and using units) until you disconnect it.")
        return
    print("Saving Google Drive and disconnecting the runtime so it stops using compute units...", flush=True)
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()
    except Exception as exc:
        print("Drive flush:", exc)
    runtime.unassign()

class guard:
    # Wrap a step: on an error (not a manual stop) print it and shut the runtime down.
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        return self
    def __exit__(self, et, ev, tb):
        if et is not None and not issubclass(et, KeyboardInterrupt):
            traceback.print_exception(et, ev, tb)
            stop_runtime(f"step '{self.name}' failed: see the error above")
        return False

print("settings ok")

## 1. Setup

In [ ]:
with guard("setup"):
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(WORK, exist_ok=True)

    # the challenge toolkit + benchmarks, pinned to the tested version
    if not os.path.isdir(os.path.join(CHALLENGE_DIR, ".git")):
        sh(f"git clone -q {CHALLENGE_URL} {CHALLENGE_DIR}")
    sh(f"git -C {CHALLENGE_DIR} checkout -q {CHALLENGE_COMMIT}")

    # your repo (always a fresh copy, so the latest pushed code is used)
    token = None
    try:
        from google.colab import userdata
        token = userdata.get(GITHUB_TOKEN_SECRET)
    except Exception:
        pass  # no secret: the repo must be public
    url = GAT_REPO_URL.replace("https://", f"https://{token}@") if token else GAT_REPO_URL
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    try:
        sh(f"git clone -q --depth 1 --branch {GAT_REPO_BRANCH} {url} {REPO_DIR}",
           show=f"git clone --branch {GAT_REPO_BRANCH} {GAT_REPO_URL} (token: {'yes' if token else 'no'})")
    except RuntimeError:
        raise RuntimeError("Could not clone your repo. Check GAT_REPO_URL / GAT_REPO_BRANCH; "
                           "if the repo is private, add the GitHub token as a Colab secret "
                           f"named {GITHUB_TOKEN_SECRET} and allow this notebook to use it.")
    CODE_DIR = os.path.join(REPO_DIR, GAT_REPO_SUBDIR)
    if not os.path.isfile(os.path.join(CODE_DIR, "gat_order", "train.py")):
        raise RuntimeError(f"gat_order/train.py not found under {CODE_DIR}: set GAT_REPO_SUBDIR "
                           "to the folder that contains gat_order/")
    ENV["PYTHONPATH"] = CODE_DIR + os.pathsep + ENV.get("PYTHONPATH", "")

    sh(f"{sys.executable} -m pip -q install torch_geometric==2.8.0.post1")
    import torch
    WORKERS = os.cpu_count() or 1
    GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    print(f"\nGPU: {GPU}   CPU cores: {WORKERS}   torch {torch.__version__}")

if GPU is None:
    print("No GPU: fine for gat_order (the network is small; the time goes into routing, on the CPU).")

## 2. Self-test
Runs the router check, pool generation, training, resume and evaluation at tiny scale. If anything is wrong with the environment it fails here, not hours into training.

In [ ]:
with guard("self-test"):
    if RUN_SELFTEST:
        sh(f"{sys.executable} -m gat_order.selftest --workers {WORKERS}", cwd=CODE_DIR)
    else:
        print("skipped")

## 3. Training data
The hard boards (up to 110 nets) of the existing `gat_mcts` pool if there is one, else a new pool of hard-recipe boards generated once into Drive. Either way the boards are copied to the local disk for speed.

In [ ]:
with guard("training data"):
    def hard_entries(pool_dir):
        p = os.path.join(pool_dir, "index.json")
        if not os.path.exists(p):
            return []
        return [e for e in json.load(open(p)) if e.get("tier", "hard") == "hard" and e["n_nets"] <= 110]
    src = EXISTING_POOL
    entries = hard_entries(src)
    if entries:
        print(f"using {len(entries)} hard boards from {src}")
    else:
        src = f"{WORK}/data"
        if len(hard_entries(src)) < 0.9 * N_TRAIN_BOARDS:
            sh(f"{sys.executable} -m gat_order.gen_data --out {src} --n {N_TRAIN_BOARDS} "
               f"--workers {WORKERS}", cwd=CODE_DIR)
        entries = hard_entries(src)
    shutil.rmtree(DATA_LOCAL, ignore_errors=True)
    os.makedirs(DATA_LOCAL)
    for e in entries:
        shutil.copy(os.path.join(src, e["name"] + ".json"), DATA_LOCAL)
    json.dump(entries, open(os.path.join(DATA_LOCAL, "index.json"), "w"), indent=1)
    print("boards:", len(entries))

## 4. Train
Resumes automatically if this run already has a checkpoint. There is no hour limit: it stops when the evaluation score hasn't improved for `PLATEAU_EVALS` evaluations. Each step line shows the mean and best score of that step's routings (the default order = 1.0) and how many never became legal; every 10 steps it scores 2 hard-tier cases.

In [ ]:
with guard("training"):
    RUN_DIR = f"{WORK}/{RUN_NAME}"
    os.makedirs(RUN_DIR, exist_ok=True)
    latest = os.path.join(RUN_DIR, "latest.pt")
    resume = f"--resume {latest}" if os.path.exists(latest) else ""
    print("resuming from", latest) if resume else print("starting a new run")
    sh(f"{sys.executable} -m gat_order.train --data {DATA_LOCAL} --out {RUN_DIR} {resume} "
       f"--workers {WORKERS} --eval-every 10 --eval-cases 2 "
       f"--plateau-evals {PLATEAU_EVALS}", cwd=CODE_DIR)

## 5. Score the hard tier
Routes all 9 hard-tier cases with the best checkpoint from training (`best.pt`, else `latest.pt`): the GAT's order plus noisy versions of it, the best legal routing kept. Writes a leaderboard-format submission to Drive, scored by the challenge's own scorer.

In [ ]:
with guard("evaluation"):
    SUB_DIR = f"{WORK}/submissions/hard/{RUN_NAME}"
    CKPT = f"{RUN_DIR}/best.pt" if os.path.exists(f"{RUN_DIR}/best.pt") else f"{RUN_DIR}/latest.pt"
    if EVALUATE:
        print("scoring with", CKPT)
        sh(f"{sys.executable} -m gat_order.evaluate --tier hard --ckpt {CKPT} "
           f"--samples {EVAL_SAMPLES} --workers {WORKERS} --out {SUB_DIR}", cwd=CODE_DIR)
    else:
        print("skipped")

## 6. Report, then shut down

In [ ]:
with guard("report"):
    import matplotlib.pyplot as plt
    rows = [json.loads(l) for l in open(os.path.join(RUN_DIR, "log.jsonl"))]
    steps = [r for r in rows if r.get("kind") == "step"]
    if steps:
        x = [r["step"] for r in steps]
        fig, ax = plt.subplots(1, 3, figsize=(16, 4))
        ax[0].plot(x, [r["mean_score"] for r in steps], label="mean", marker=".")
        ax[0].plot(x, [r["best_score"] for r in steps], label="best", alpha=0.6)
        ax[0].axhline(1.0, color="gray", lw=1, ls="--")
        ax[0].set_title("score of the step's routings (default order = 1.0)"); ax[0].legend()
        ax[1].plot(x, [r["failed"] / r["routings"] for r in steps], marker=".")
        ax[1].set_title("share of routings that never became legal")
        ax[2].plot(x, [r["spread"] for r in steps])
        ax[2].set_title("spread of the GAT's scores (vs noise sigma)")
        for a in ax:
            a.set_xlabel("step")
        plt.show()
    for r in rows:
        if r.get("kind") == "eval":
            print(f"step {r['step']}: " + ", ".join(
                f"{c['case']} greedy {c['greedy']:.4f} best {c['best']:.4f}" for c in r["cases"]))
    if os.path.exists(f"{RUN_DIR}/best.json"):
        print("best checkpoint:", json.load(open(f"{RUN_DIR}/best.json")))
    print(f"\ncheckpoints + log: {RUN_DIR}")
    if EVALUATE:
        print(f"submission:        {SUB_DIR}")

stop_runtime("All done")